In [ ]:

Fine-tune **Qwen2.5-7B-Instruct** bằng **QLoRA + Unsloth** trên Google Colab miễn phí (T4 15GB).

Mục tiêu:
1. Dạy LLM tạo câu hỏi TOEIC/IELTS đúng format chuẩn
2. Dạy LLM giải thích đáp án theo phong cách thầy cô Việt Nam

**Yêu cầu**: Chọn Runtime > Change runtime type > **T4 GPU**

## Bước 1: Cài thư viện

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps unsloth

: 

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded successfully!")

## Bước 3: Cấu hình LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Bước 4: Chuẩn bị dataset

Upload file `finetune_dataset.jsonl` lên Colab hoặc Google Drive.

File được tạo bằng script: `python scripts/generate_finetune_dataset.py`

In [ ]:
import os
from google.colab import files

DATASET_PATH = "finetune_dataset.jsonl"

if not os.path.exists(DATASET_PATH):
    print("Upload file finetune_dataset.jsonl...")
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()
print(f"Số mẫu trong dataset: {len(lines)}")

In [ ]:
import json
from datasets import Dataset

def format_chat(sample):
    """Chuyển conversations sang chat template của Qwen."""
    messages = sample["conversations"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

raw_data = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

print(f"Dataset size: {len(dataset)}")
print(f"\nVí dụ mẫu đầu tiên:\n{dataset[0]['text'][:500]}...")

## Bước 5: Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

print("Bắt đầu training...")
stats = trainer.train()
print(f"\nTraining hoàn tất!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Thời gian: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
used = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total = round(gpu_stats.total_mem / 1024**3, 2)
print(f"VRAM sử dụng: {used} GB / {total} GB ({100*used/total:.1f}%)")

## Bước 6: Test model sau fine-tune

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Tạo 3 câu hỏi TOEIC Reading Part 5 trình độ intermediate.",
    "Giải thích tại sao trong câu 'She ___ here since 2020', đáp án đúng là 'has worked' chứ không phải 'worked'. Giải thích theo phong cách thầy cô Việt Nam.",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print("=" * 60)
    print(f"Prompt: {prompt[:80]}...")
    print(f"\nResponse:\n{response}")
    print()

## Bước 7: Lưu model

### Option A: Lưu LoRA adapter lên HuggingFace

In [ ]:
# Đổi thành username HuggingFace của bạn
HF_USERNAME = "LeNhan18"
MODEL_NAME = f"{HF_USERNAME}/ViEng-Qwen2.5-7B-lora"

# Lưu local trước
model.save_pretrained("vieng-lora-adapter")
tokenizer.save_pretrained("vieng-lora-adapter")
print("Đã lưu adapter vào vieng-lora-adapter/")

# Push lên HuggingFace (cần login trước)
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# model.push_to_hub(MODEL_NAME)
# tokenizer.push_to_hub(MODEL_NAME)
# print(f"Đã push lên https://huggingface.co/{MODEL_NAME}")

### Option B: Export sang GGUF (chạy local với llama.cpp / Ollama)

In [ ]:
# Merge LoRA vào base model rồi export GGUF 4-bit
# (cần thêm RAM, có thể cần Colab Pro)

# model.save_pretrained_gguf(
#     "vieng-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# print("Đã export GGUF!")

# Download file GGUF về máy
# from google.colab import files
# files.download("vieng-gguf/unsloth.Q4_K_M.gguf")

### Option C: Lưu adapter vào Google Drive (đơn giản nhất)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_path = "/content/drive/MyDrive/ViEng/vieng-lora-adapter"
shutil.copytree("vieng-lora-adapter", save_path, dirs_exist_ok=True)
print(f"Đã lưu adapter vào Google Drive: {save_path}")